# Set up

## Check configuration
Should return path to correct python version (from virtual environment)

In [ ]:
import sys

print(sys.executable)

## Load libraries

In [ ]:
# Automatically reload modules before execution of each cell
# so when you edit src/mypackage/*.py in your editor and rerun cells,
# changes appear immediately.
%reload_ext autoreload
%autoreload 2

# python
from __future__ import annotations

# Standard library
from pathlib import Path

# Third-party
import pandas as pd
from dotenv import find_dotenv, load_dotenv

# Custom
## Functions
from tidysdmx import (
    build_structure_map_from_template_wb,
    collect_structure_map_artifacts,
    create_schema_from_table,
    map_structures,
    parse_mapping_template_wb,
    sanitize_variable,
    standardize_output,
    validate_dataset_local,
)

## FMR read/write workflow.
## NOTE: these live in `tidysdmx.fmr`, not the top-level package.
from tidysdmx.fmr import (
    ChangeImpact,
    FmrClient,
    StructureAction,
    VersionPolicy,
    execute_plan,
    inplace_breaking_actions,
    plan_publication,
    plan_to_dataframe,
    rebase_to_registry,
    report_to_dataframe,
)

## Define globals

In [ ]:
# FMR registry
fmr_url = "https://fmrqa.worldbank.org/FMR/sdmx/v2"

# --- Target: the published dissemination structure the mapping writes into ---
dis_structure_agency = "WB.GGH.HSP"
dis_structure_id = "DS_ASPIRE"
dis_structure_version = "1.0.0"
target_artefact_id = f"{dis_structure_agency}:{dis_structure_id}({dis_structure_version})"

# --- Source: the tidy_raw schema this notebook builds in STEP 3 ---
# This schema is scratch: it only describes THIS notebook's raw input, and STEP 9
# replaces it in place on every run. We DON'T pin a version by hand here -- STEP 9
# seeds it from the registry (see rebase_to_registry in 9.1b). The value below is
# only the version used the FIRST time the schema is created.
#
# Use a DRAFT semver ("1.0.0-draft"): three numeric parts make it proper SDMX 3.0
# (unlike the two-part "1.0", which silently degrades the version policy), while the
# "-draft" extension makes it NON-final -- so under VersionPolicy(replace_non_final=
# True) it is overwritten in place rather than bumped. That is exactly the scratch
# behaviour we want, without the two-part downgrade. The dissemination artefacts
# (WB.GGH.HSP) are final semver and bump normally.
src_agency = "WB.DP"
src_schema_id = "DP_SCHEMA"
SRC_DRAFT_VERSION = "1.0.0-draft"

# Path to raw data
path_to_raw_data = Path("./data/WB_ASPIRE")
path_to_xlsx_mapping = Path("./data/WB_ASPIRE_MAPPING_SUBSET.xlsx")

# FMR write credentials.
#
# A .env file is just a text file -- nothing reads it automatically. FmrClient
# reads os.environ, so the .env has to be loaded into the process environment
# first. find_dotenv() walks up from this notebook's directory to the repo root.
#
# Put these in your .env (never hard-code them here, and never commit it):
#   TIDYSDMX_FMR_USER=...
#   TIDYSDMX_FMR_PASSWORD=...        (or TIDYSDMX_FMR_TOKEN=...)
dotenv_path = find_dotenv()
assert dotenv_path, "No .env found. Create one at the repo root (see above)."
load_dotenv(dotenv_path, override=True)
print(f"Loaded credentials from: {dotenv_path}")

## Initiate FMR client and load required SDMX artefacts

### Initiate FMR API client

In [ ]:
# FmrClient is a read/write facade over pysdmx's registry clients.
# Reads need no credentials; the write client is lazy and is not built (nor are
# credentials checked) until STEP 9 actually publishes.
client = FmrClient(fmr_url)
client.api_endpoint

### Fetch dissemination schema

In [ ]:
dis_schema = client.get_schema(
    "datastructure",
    agency=dis_structure_agency,
    id=dis_structure_id,
    version=dis_structure_version,
)
dis_schema

# STEP 1: Load raw data

Here we are loading the raw dataset as provided from the source. In this demonstration notebook, the raw data is simply being loaded from file, but in the final pipeline, the provenance of the file should be fully documented in a configuration file, and read from the source / DDH possible.

In [ ]:
def read_raw_data(folder_path):
    """Read all .csv files from a folder and return a single pandas DataFrame.

    Parameters
    ----------
    folder_path : str | pathlib.Path
        Path to the folder containing CSV files.

    Returns:
    -------
    pandas.DataFrame
        Concatenated DataFrame of all CSVs (same structure assumed), with a
        "Series code" column indicating the source filename (without .csv).
    """
    folder = Path(folder_path)
    csv_files = sorted(folder.glob("*.csv"))
    if not csv_files:
        raise ValueError(f"No CSV files found in folder: {folder}")

    dfs = []
    for f in csv_files:
        df = pd.read_csv(f)
        df["Series code"] = f.stem  # filename without extension
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


raw_df = read_raw_data(path_to_raw_data)
raw_df.head()

# STEP 2: Reshape raw data

A critical step of this opinionated pipeline framework is to systematically reshape raw into tidy format (one observation per row). For more information about tidy data, please refer to [Hadley Wickham's original paper](https://vita.had.co.nz/papers/tidy-data.pdf).

This step is critical because once data has been reshaped into a tidy format, the rest of the pipeline can be fully standardized, bringing immediate maintenance, scalability, and institutional knowledge benefits.

This is also a good place to implement minimal data cleaning if necessary.

In [ ]:
def reshape_raw_data(df: pd.DataFrame) -> pd.DataFrame:
    """Reshape raw data and implements basic data cleaning.

    It 'melts' (unpivots) columns starting with 'data.' into two columns ('name' and 'value'),
    and then cleans the 'name' column by removing the 'data.' prefix.

    Args:
        df: The input pandas DataFrame containing columns like 'data.1', 'data.2', etc.

    Returns:
        A new DataFrame in the longer format.
    """
    # 1. Equivalent of R's pivot_longer (using melt)
    # Selects columns starting with 'data.' for unpivoting
    data_cols = df.filter(like="YR").columns.tolist()

    df_lg = df.melt(
        id_vars=[
            col for col in df.columns if col not in data_cols
        ],  # Keep all non-data columns as identifier variables
        var_name="year",  # New column for the original column names
        value_name="value",  # New column for the values
    )

    # 2. Equivalent of R's stringr::str_replace
    # Removes the 'data.' prefix from the 'name' column
    df_lg["year"] = df_lg["year"].str.replace("YR", "", regex=False)

    # Filter out rows with missing values in 'value'
    df_lg = df_lg.dropna(subset=["value"])

    # Rename 'economy' column to 'Series' (expected in mapping template)
    # Rename year to TIME_PERIOD to match SDMX convention
    df_lg = df_lg.rename(columns={"Series code": "Series", "year": "TIME_PERIOD"})

    # Turn all headers to uppercase (to match SDMX concept IDs convention)
    df_lg.columns = [col.upper() for col in df_lg.columns]

    return df_lg


tidy_raw_df = reshape_raw_data(raw_df)
tidy_raw_df.head()

## 2.1 Sanitize variables to ensure they are SDMX compliant
Some characters are invalid for code IDs in SDMX, so variables need to be sanitized to ensure those characters are not present

In [ ]:
dimensions = ["SERIES", "ECONOMY"]
tidy_raw_df[dimensions] = tidy_raw_df[dimensions].map(sanitize_variable)

tidy_raw_df.head()

# STEP 3: Describe the tidy raw data input

We will describe the tidy raw data input using an SDMX schema. This description will allow for early validation of our input data during subsequent runs of the pipeline for data updates.

The `create_schema_from_table()` helper function builds the DSD, concept scheme and codelists automatically, with minimal input from the pipeline developer.

These artefacts are published to the FMR in STEP 9, and the structure map built in STEP 6 points at the DSD created here — which is why the agency/id/version are passed explicitly from the globals rather than left to default.

In [ ]:
# Build the tidy_raw DSD / concept scheme / codelists from the data. The version
# is the DRAFT initial (SRC_DRAFT_VERSION) -- only used the first time these are
# created; on later runs STEP 9.1b reseeds it from the registry.
tidy_raw_schema = create_schema_from_table(
    tidy_raw_df,
    dimensions=dimensions,
    time_dimension="TIME_PERIOD",
    measure="VALUE",
    agency_id=src_agency,
    schema_id=src_schema_id,
    version=SRC_DRAFT_VERSION,
)

# Derive the structure map's source ref from the DSD we just built, so the two
# cannot drift apart (same agency / id / version, in AGENCY:ID(VERSION) form).
_dsd = tidy_raw_schema.dsd
_src_agency = getattr(_dsd.agency, "id", _dsd.agency)
source_artefact_id = f"{_src_agency}:{_dsd.id}({_dsd.version})"

tidy_raw_schema.dsd.to_schema()

# STEP 4: Filter out unnecessary rows (Optional)

In [ ]:
def apply_constraints(df: pd.DataFrame, constraints: dict[str, list]) -> pd.DataFrame:
    """Filters a DataFrame based on a dictionary of column names and valid values.

    Args:
        df (pd.DataFrame): The source dataframe.
        constraints (dict): A dict where keys are column names and values are
                        lists of valid entries to keep (e.g., {'col': ['val1', 'val2']}).

    Returns:
        pd.DataFrame: A filtered copy of the original dataframe.
    """
    for column, valid_values in constraints.items():
        # strict check: ensure column exists to avoid KeyErrors
        if column in df.columns:
            df = df[df[column].isin(valid_values)]
        else:
            print(f"Warning: Column '{column}' not in DataFrame. Skipping.")

    return df


# Adapt these to your source. Each key is a column, each value the rows to keep.
constraints = {
    "SERIES": ["PER_ALLSP_ADQ_EP_PRET_TOT", "PER_ALLSP_ADQ_EP_TOT"],
}

tidy_raw_df = apply_constraints(tidy_raw_df, constraints)
tidy_raw_df

# STEP 5: Validate tidy raw data (Optional)
This step should only be required in production when we want to ensure that the ingested raw data still meets our expected format requirements (as described in tidy_raw_schema). During pipeline development, the tidy_raw_schema is inferred from the ingested raw data, so the validation should, in theory, always be successfull. This is therefore an extra-cautionary step to ensure that everything is working as expected.

In [ ]:
raw_schema = tidy_raw_schema.dsd.to_schema()
raw_errors = validate_dataset_local(df=tidy_raw_df, schema=raw_schema, sdmx_cols=[])
raw_errors

# STEP 6: Create structure map

## Create structure map from mapping template

In [ ]:
# `source_artefact_id` is derived in STEP 3 from the built tidy_raw DSD (its draft
# version), and `target_artefact_id` is the dissemination structure from the setup
# cell. The map and its representation maps initialise at the INFO-sheet version
# (WB.GGH.HSP final semver 1.0.0); STEP 9.1b reseeds them from the registry on
# re-runs, so that INFO version only matters the first time they are created.
mappings = parse_mapping_template_wb(path_to_xlsx_mapping)
sm = build_structure_map_from_template_wb(
    mappings,
    target_structure_id=target_artefact_id,
    source_structure_id=source_artefact_id,
)
sm.maps

# STEP 7: Map data to dissemination schema

## 7.1 Implement mapping

In [ ]:
mapped = map_structures(df=tidy_raw_df, structure_map=sm, verbose=True)
mapped

## 7.2 Standardize output for upload

In [ ]:
out = standardize_output(
    df=mapped, artefact_id=target_artefact_id, schema=dis_schema, action="I"
)

out

# STEP 8: Final validation

In [ ]:
dis_errors = validate_dataset_local(df=out, schema=dis_schema)
dis_errors

# STEP 9: Publish artefacts to the FMR

Everything so far has been local. This step pushes the SDMX artefacts this
notebook built — the concept scheme, codelists, DSD, structure map and its
representation maps — into the FMR.

The workflow is **plan → rehearse → publish**:

1. **Assemble** every artefact into a *single* batch (9.1).
2. **Plan** it against the registry (9.2). `plan_publication` only *reads*: it
   fetches each artefact's registry copy, diffs it, and proposes CREATE / UPDATE
   / SKIP with a version for each. Re-run it as often as you like — it is the
   "what would change if I published?" step, and it is the one you will live in
   during development.
3. **Publish** (9.3–9.4), first as a dry run, then for real.

Finally, 9.5 **finalizes** the draft when development is done, which is what
makes the artefacts safe for the production pipeline to consume.

## 9.1 - Assemble the publication batch

Two things to know here, both of which will bite you otherwise.

**Publish everything in ONE batch.** The structure map's *source* is the DSD
built in STEP 3, so they depend on each other. When a dependency's version is
bumped, `plan_publication` rewrites the references of artefacts **in the same
call** — it cannot rewrite references it has not been handed. Splitting the DSD
and the structure map into two `plan_publication` calls would leave the map
pointing at a stale DSD version, silently. Order within the list does not
matter: the planner sorts by dependency layer itself.

**Pass `convert_to_urns=False`.** By default `collect_structure_map_artifacts`
replaces embedded `RepresentationMap` objects with URN strings — which is what
the SDMX-ML *file* writer needs, but publish-readiness validation rejects it
(rule SM003: *"references an unresolved RepresentationMap URN; embed the object
instead"*). Keeping the objects embedded is also correct on the wire: pysdmx's
SDMX-JSON writer converts them to URN references when it serialises the
submission. For the same reason, do **not** use `prepare_structure_map_for_upload`
here — it hard-codes the URN conversion and its output is rejected.

In [ ]:
publish_batch = [
    tidy_raw_schema.concept_scheme,
    *tidy_raw_schema.codelists,
    tidy_raw_schema.dsd,
    # convert_to_urns=False keeps RepresentationMaps embedded -- see note above.
    *collect_structure_map_artifacts(sm, convert_to_urns=False),
]

print(f"Publication batch: {len(publish_batch)} artefacts")
for artefact in publish_batch:
    print(f"  - {artefact.__class__.__name__}: {artefact.short_urn}")

## 9.1b - Reconcile versions with the registry (read-only)

Every artefact above was built at a *fixed* initial version (the tidy_raw draft;
the mapping template's INFO version). On a re-run months later the registry has
moved on, so those fixed versions are stale — which makes `plan_publication` raise
a blocking `P002` ("registry newer than local baseline") for the final artefacts,
and would silently overwrite the scratch draft.

`rebase_to_registry` fixes that in one read-only pass: it **seeds each artefact's
version from the registry** (existing → the registry's latest; new → its initial),
and retargets the intra-batch references to match. After this, the plan in 9.2
diffs against the registry's own baseline and infers the correct published version.

In [ ]:
# Seed every artefact's version from the registry BEFORE planning, so the plan
# diffs against the registry's own baseline instead of the version we happened to
# build locally.
#
# For each artefact: if it already exists in the FMR, adopt the registry's latest
# version; if it does not, keep its build-time initial version. Intra-batch
# references (the structure map's source + its embedded representation maps) are
# retargeted to match. Effect:
#   - WB.GGH.HSP dissemination artefacts (final semver) -> the false "registry
#     newer than local baseline" (P002) block disappears and bumps are computed
#     from the true current version;
#   - the WB.DP tidy_raw draft -> resolves to an in-place replace at its draft
#     version (no version sprawl).
#
# ONE-TIME MIGRATION: if the registry still holds an OLD two-part `WB.DP:DP_SCHEMA
# (1.0)` from a previous version of this notebook, delete the stale WB.DP scratch
# artefacts (DP_SCHEMA, DP_SCHEMA_CS, CL_*) from the FMR once before the first run.
# Otherwise rebase would seed the draft back down to `1.0` and never create the
# `1.0.0-draft`. They are scratch with no consumers, so deletion is safe.
publish_batch = rebase_to_registry(client, publish_batch)

print("Seeded versions (from the registry, or the build-time initial if new):")
for artefact in publish_batch:
    print(f"  - {artefact.__class__.__name__}: {artefact.short_urn}")

## 9.2 - Plan the publication (read-only, safe to re-run)

This cell writes nothing to the registry. It is the step to iterate on: change
your mapping template or your raw data, re-run, and read off exactly what would
land in the FMR.

`plan_to_dataframe` gives one row per artefact — the action (CREATE / UPDATE /
SKIP), the version in the registry, the version that would be published, the
change impact, and any blocking issues.

**Read the plan before you publish.** Two things to check every time:

- **The version each artefact resolves to.** The planner always diffs against
  whatever is *latest* in the registry, and the version you built locally is not
  compared — so the proposed version comes from the registry, not from your
  code. If you expected a new version and see the old one, that is why.
- **Anything marked `breaking`.** Under `DEV_POLICY` the tidy_raw scratch schema
  is replaced *in place*, breaking changes included. That is fine for scratch and
  wrong for anything published — so if a `WB.GGH.HSP` artefact ever shows
  `breaking`, stop and look at it.

In [ ]:
# Development policy. `replace_non_final=True` republishes anything that is not
# final at the SAME version instead of bumping it.
#
# In practice that splits the batch exactly the way you want:
#   - the WB.DP tidy_raw scratch schema is at the two-part version "1.0", which
#     is never final under SDMX 3.0 -> overwritten in place, no version sprawl;
#   - the WB.GGH.HSP dissemination artefacts are semver and final -> they bump
#     normally (e.g. 1.0.0 -> 1.0.1 for a cosmetic change).
DEV_POLICY = VersionPolicy(replace_non_final=True)

plan = plan_publication(
    client,
    publish_batch,
    policy=DEV_POLICY,
    action=StructureAction.Replace,
)

print(plan.summary())
plan_df=plan_to_dataframe(plan)

## 9.2b - Guard against a silent breaking overwrite

The tidy_raw scratch schema is a **draft**, so it is replaced *in place* every run —
which is fine for a cosmetic or additive change, but a *breaking* change (a dropped
or retyped dimension after an upstream format change) would overwrite the previous
draft with an incompatible structure and no version to tell them apart.

This cell stops the pipeline if any artefact would be overwritten in place with a
breaking diff, and prints what changed. Review it; if the re-development is
intended, acknowledge and re-run.

In [ ]:
# The WB.DP tidy_raw scratch schema is republished IN PLACE at its draft version
# (no bump), so a breaking change would silently overwrite the previous draft.
# `inplace_breaking_actions` returns exactly the actions that overwrite an artefact
# in place with a breaking diff. Surface them and require explicit acknowledgement
# before publishing.
ACK_TIDY_RAW_BREAKING = False  # set True only after reviewing the diffs below

breaking = inplace_breaking_actions(plan)
if breaking:
    print("BREAKING in-place overwrites detected:")
    for action in breaking:
        print(f"\n{action.short_urn}:")
        print(action.diff.summary())
    if not ACK_TIDY_RAW_BREAKING:
        raise RuntimeError(
            "Refusing to overwrite a non-final (draft) artefact with a breaking "
            "change. Review the diffs above; if this is an intended re-development "
            "of the scratch schema, set ACK_TIDY_RAW_BREAKING=True and re-run."
        )
else:
    print("No breaking in-place overwrites. Safe to publish.")

## 9.3 - Check write credentials

Set `TIDYSDMX_FMR_USER` and `TIDYSDMX_FMR_PASSWORD` in your environment before
running this (or `TIDYSDMX_FMR_TOKEN`). Never put credentials in the notebook or
in a file next to it.

In [ ]:
# Everything up to here is read-only. Publishing needs write credentials, which
# FmrClient reads from TIDYSDMX_FMR_USER / TIDYSDMX_FMR_PASSWORD (or
# TIDYSDMX_FMR_TOKEN). The write client is lazy, so touching it here surfaces a
# missing credential now rather than half-way through a publish.
_ = client.writer
print(f"Write client ready for: {client.api_endpoint}")

## 9.4 - Rehearse, then publish

`execute_plan` refuses to touch the network at all if the plan has blocking
issues, so a bad artefact fails fast and locally.

Run the dry run first and read `report.summary()`.

> **Careful:** a dry run marks every action as *skipped*, and `report.ok` counts
> "skipped" as fine — so `report.ok` is `True` for **any** dry run. It is not a
> "did it publish?" signal. Read the summary, not the flag.

In [ ]:
# Dry run: validates and resolves everything, but sends nothing.
report = execute_plan(client, plan, dry_run=True)
print(report.summary())

In [ ]:
# For real. Re-run 9.2 first if you changed anything since planning.
report = execute_plan(client, plan, dry_run=False)
print(report.summary())
report_to_dataframe(report)

## 9.5 - Hand off to production

The production pipeline does not build artefacts — it reads them from the FMR.
So the last thing development owes it is a note of *what to point at*.

Run this after a successful publish to list the versions now in the registry.
Those are the versions the production pipeline should consume.

The tidy_raw scratch schema (`WB.DP`) is deliberately not in that list: it
describes this notebook's raw input and is overwritten on every run. Production
consumes the dissemination artefacts (`WB.GGH.HSP`) — the structure map, its
representation maps, and the dissemination DSD.

In [ ]:
# Re-plan read-only: with everything published, every action should now be SKIP.
# Anything still showing CREATE/UPDATE did not make it into the registry.
#
# Re-seed from the registry first: publishing bumped the WB.GGH.HSP artefacts, so
# the in-memory publish_batch still holds the pre-publish versions. Rebasing adopts
# the just-published versions, so a clean handoff shows all SKIP.
handoff_batch = rebase_to_registry(client, publish_batch)
handoff = plan_publication(
    client,
    handoff_batch,
    policy=DEV_POLICY,
    action=StructureAction.Replace,
)
print(handoff.summary())

print("Versions for the production pipeline to consume:")
for action in handoff.actions:
    if action.artefact.agency != src_agency:
        print(f"  {action.short_urn}")